# AegisRAG · Stage 03 — Generator SFT (QLoRA + DoRA)
    Supervised fine-tune of `Qwen/Qwen2.5-7B-Instruct` with NF4 4-bit quant, DoRA adapters, citation-weighted CE.
    **Resource notes**
    * Fits on 1× T4 (16 GB) at `batch_size=2, grad_accum=16, max_seq_length=1024`.
    * For faster convergence prefer a 2× T4 kernel and use accelerate or torchrun.
    * Expect ~3–4 h per epoch at 5k QA pairs. Kaggle session cap is 12 h — aim for 1 epoch then resume.
    Attach: `aegisrag-source`, `aegisrag-synthetic`.


In [ ]:
# ---- Kaggle bootstrap ----------------------------------------------------
# Attach these Kaggle Datasets to the notebook before running:
#   aegisrag-source       (snapshot of the git repo: src/, config/, run.py, requirements.txt)
#   aegisrag-raw-docs     (raw customer-support documents)            [only for data-gen]
#   aegisrag-synthetic    (generated qa/preferences/labels .jsonl)    [all training stages]
#   aegisrag-checkpoints  (prior-stage checkpoints)                   [required for DPO]
import sys, os, importlib.util
spec = importlib.util.spec_from_file_location(
    "aegisrag_bootstrap",
    "/kaggle/input/aegisrag-source/kaggle/helpers/bootstrap.py",
)
bootstrap = importlib.util.module_from_spec(spec); spec.loader.exec_module(bootstrap)
bootstrap.setup_kaggle(
    component='generator_sft',
    install_train_deps=True,
    install_parse_deps=False,
)


In [ ]:
# Tight fit for 1× T4: reduce max_seq_length and batch size.
import os
os.environ["AEGIS_TRAINING__GENERATOR_SFT__MAX_SEQ_LENGTH"] = "1024"
os.environ["AEGIS_TRAINING__GENERATOR_SFT__BATCH_SIZE"] = "2"
os.environ["AEGIS_TRAINING__GENERATOR_SFT__GRADIENT_ACCUMULATION_STEPS"] = "16"
os.environ["AEGIS_TRAINING__FP16"] = "true"
os.environ["AEGIS_TRAINING__BF16"] = "false"  # T4 lacks bf16 tensor cores
os.environ["AEGIS_TRAINING__GRADIENT_CHECKPOINTING"] = "true"

from src.utils.config import reset_config
reset_config()


In [ ]:
from src.training.train_generator import train
result = train()
result


In [ ]:
# ---- Persist outputs -----------------------------------------------------
# Everything under /kaggle/working is captured as the notebook's output and
# can be saved as a new Kaggle Dataset via File > "Save Version"
# with output = "Save & Run All (Commit)".
import shutil, os
from pathlib import Path
src_dir = Path("/kaggle/working/checkpoints/generator_sft")
print("SFT adapter:", src_dir, "->", os.listdir(src_dir) if src_dir.exists() else "(missing)")
